### Imports, Parameters 

In [1]:
import numpy as np
import os
import subprocess
import matplotlib.pyplot as plt
from ase.io import read
from ase.build import bulk, make_supercell
import warnings
warnings.filterwarnings('ignore')

import ase
print(f'ASE version : {ase.__version__}')
print(f'NumPy       : {np.__version__}')

# ═══════════════════════════════════════════════════════════
# YOUR SYSTEM PATHS
# ═══════════════════════════════════════════════════════════
LAMMPS_CMD   = '/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap/lmp'
MACE_MODEL   = '/projects/westgroup/akinyemi.az/mace_lammps/models/mace-mp-0b2-medium.model-mliap_lammps.pt'
KOKKOS_FLAGS = ['-k', 'on', 'g', '1', '-sf', 'kk',
                '-pk', 'kokkos', 'newton', 'on', 'neigh', 'half']
SLURM_CONFIG = {
    'partition'     : 'multigpu',
    'ntasks'        : 1,
    'cpus_per_task' : 8,
    'gpu'           : 'a100:1',
    'time'          : '24:00:00',
    'conda_env'     : 'mace-lammps',
    'cuda_version'  : '12.3.0',
    'openmpi_ver'   : '4.1.6',
    'ld_paths'      : [
        '/shared/EL9/explorer/cuda/12.3.0/lib64/stubs',
        '/projects/westgroup/akinyemi.az/mace_lammps/lammps/build-mliap',
        '/home/akinyemi.az/miniforge3/envs/mace-lammps/lib',
    ]
}

# ── Fixed constants ──────────────────────────────────────────
ELEM_STR    = 'Al B C Cr Fe Mo Ni O H'
PAIR_STYLE  = 'mliap unified'
PAIR_SUFFIX = '0'
E2T = {'Al':1,'B':2,'C':3,'Cr':4,'Fe':5,'Mo':6,'Ni':7, 'O':8, 'H':9}
MASSES = {
    1:(26.9815,'Al'), 2:(10.8110,'B'),  3:(12.0110,'C'),
    4:(51.9961,'Cr'), 5:(55.8450,'Fe'), 6:(95.9600,'Mo'),
    7:(58.6934,'Ni'), 8:(15.9990, 'O'), 9:( 1.0080,'H'),
}

# ── MD parameters ────────────────────────────────────────────
TEMPERATURES  = [300, 400, 600, 800]  # K — Arrhenius points
TIMESTEP      = 0.0005   # ps = 0.5 fs
N_EQUIL       = 500000   # steps equilibration = 50 ps
N_PROD        = 1500000   # steps production = 100 ps (written for NB11)1620000
# N_PROD        = 1620000   # steps production = 100 ps (written for NB11)
TAU_T         = 0.1      # ps — Nosé-Hoover thermostat time constant
THERMO_EVERY  = 1000     # output frequency
DUMP_EVERY    = 1000      # trajectory dump frequency for MSD
# N_EQUIL = 1000000
# N_PROD = 200000
# DUMP_EVERY = 100

# ── File paths ───────────────────────────────────────────────
BULK_MIN  ='structures/02_bulk_energy_minimization/Hastelloy_N_42_minimized.lammps'

for d in ['structures/notebook10-bulk-equilibration/N_42', 'results/notebook10-bulk-equilibration/N_42', 'lammps_scripts/notebook10-bulk-equilibration/N_42', 'slurm_scripts/notebook10-bulk-equilibration/N_42']:
    os.makedirs(d, exist_ok=True)

# ── Read a₀ from NB02 ─────────────────────────────────────────
# Use minimized bulk structure to get correct lattice parameter
if not os.path.exists(BULK_MIN):
    raise FileNotFoundError(f'{BULK_MIN} not found. Complete NB02 first.')

bulk_slab = read(BULK_MIN, format='lammps-data', style='atomic')
A0 = bulk_slab.cell.lengths()[0] / 5  # 5x5x5 supercell → divide by 5
print(f'a₀ from minimized bulk: {A0:.6f} Å  (NB02)')

# # ── Read best subsurface site type from NB08 ─────────────────
# BEST_SUB_LABEL = 'O00'  # default fallback
# if os.path.exists('results/notebook08/subsurface_energies.txt'):
#     with open('results/notebook08/subsurface_energies.txt') as f:
#         for line in f:
#             if 'Most stable:' in line:
#                 BEST_SUB_LABEL = line.split()[2]
#                 break
# print(f'Best subsurface site from NB08: {BEST_SUB_LABEL}')
# print(f'Site type: octahedral — H at body-center of FCC unit cell')
# print(f'\nMD setup:')
# print(f'  Temperatures  : {TEMPERATURES} K')
# print(f'  Timestep      : {TIMESTEP*1000:.0f} fs')
# print(f'  Equilibration : {N_EQUIL*TIMESTEP:.0f} ps')
# print(f'  Production    : {N_PROD*TIMESTEP:.0f} ps')
# print(f'  Thermostat    : Nosé-Hoover τ = {TAU_T} ps')

ASE version : 3.27.0
NumPy       : 2.4.3
a₀ from minimized bulk: 3.573441 Å  (NB02)


In [2]:
# ─────────────────────────────────────────────────────────────
# Build bulk supercell + 1 H at octahedral interstitial
# Octahedral site in FCC: body center of unit cell = (½,½,½) frac
# We place H near the geometric center of the supercell
# Refs: H in FCC [3,4]
# ─────────────────────────────────────────────────────────────

bulk_atoms = read(BULK_MIN, format='lammps-data', style='atomic')
bulk_pos   = bulk_atoms.get_positions()
bulk_syms  = np.array(bulk_atoms.get_chemical_symbols())
L          = bulk_atoms.cell.lengths()

print(f'Bulk supercell: {len(bulk_atoms)} atoms')
print(f'Cell: {L[0]:.4f} × {L[1]:.4f} × {L[2]:.4f} Å')
print(f'a₀ = {L[0]/5:.6f} Å')

# Find octahedral site near center of supercell
# FCC octahedral site at fractional (½,½,½) within each unit cell
# In the 5x5x5 supercell, the central unit cell is at (1,1,1) in cell coords
# so the central octahedral site is at fractional (1.5/5, 1.5/5, 1.5/3) = (0.3, 0.3, 0.3)
# i.e. the exact center of the supercell
cx = L[0] / 2
cy = L[1] / 2
cz = L[2] / 2

# Verify no metal atom is too close (overlap check)
h_pos = np.array([cx, cy, cz])
dists = np.linalg.norm(bulk_pos - h_pos, axis=1)
min_dist = dists.min()
print(f'\nH placed at ({cx:.4f}, {cy:.4f}, {cz:.4f}) Å')
print(f'Nearest metal atom: {min_dist:.4f} Å  (should be > 1.5 Å)')

if min_dist < 1.5:
    # Shift H slightly if too close
    h_pos = np.array([cx + 0.1, cy + 0.1, cz + 0.1])
    dists = np.linalg.norm(bulk_pos - h_pos, axis=1)
    min_dist = dists.min()
    print(f'Shifted H to avoid overlap: min dist = {min_dist:.4f} Å')

# Build combined structure
all_syms = list(bulk_syms) + ['H']
all_pos  = np.vstack([bulk_pos, h_pos])

# Write LAMMPS data file
BULK_H_FILE = 'structures/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_initial.lammps'
with open(BULK_H_FILE, 'w') as f:
    f.write(f'# Hastelloy N bulk 5x5x5 + 1H at octahedral site — NB10\n\n')
    f.write(f'{len(all_syms)} atoms\n{len(MASSES)} atom types\n\n')
    f.write(f'0.0  {L[0]:.10f}  xlo xhi\n')
    f.write(f'0.0  {L[1]:.10f}  ylo yhi\n')
    f.write(f'0.0  {L[2]:.10f}  zlo zhi\n\n')
    f.write('Masses\n\n')
    for t, (m, el) in MASSES.items():
        f.write(f'{t}  {m}  # {el}\n')
    f.write('\nAtoms # atomic\n\n')
    for i, (sym, p) in enumerate(zip(all_syms, all_pos), 1):
        f.write(f'{i}  {E2T[sym]}  {p[0]:.10f}  {p[1]:.10f}  {p[2]:.10f}\n')

print(f'\nWritten: {BULK_H_FILE}')
print(f'  Total atoms : {len(all_syms)} (500 metal + 1 H)')
print(f'  H at        : ({h_pos[0]:.4f}, {h_pos[1]:.4f}, {h_pos[2]:.4f}) Å')
print(f'  c_H         : {1/500*100:.2f} at.%  (dilute limit)')

Bulk supercell: 500 atoms
Cell: 17.8672 × 17.8672 × 17.8672 Å
a₀ = 3.573441 Å

H placed at (8.9336, 8.9336, 8.9336) Å
Nearest metal atom: 1.7373 Å  (should be > 1.5 Å)

Written: structures/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_initial.lammps
  Total atoms : 501 (500 metal + 1 H)
  H at        : (8.9336, 8.9336, 8.9336) Å
  c_H         : 0.20 at.%  (dilute limit)


In [3]:
# ─────────────────────────────────────────────────────────────
# Write LAMMPS NVT script for bulk H equilibration
# Phase 1: equilibration (50 ps) — system reaches T
# Phase 2: production (100 ps) — trajectory written for NB11 MSD
# Nosé-Hoover thermostat on all atoms (no frozen layers in bulk)
# Refs: NVT MD [6]; H diffusion [1,2]
# ─────────────────────────────────────────────────────────────

def write_bulk_nvt(T):
    traj_file = f'results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_{T}K_prod.lammpstrj'
    log_file  = f'results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_equil_{T}K.log'
    out_file  = f'structures/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_{T}K.lammps'

    script = f"""# LAMMPS NVT bulk equilibration — Hastelloy N + 1H
# T = {T} K | Notebook 10
# Phase 1: {N_EQUIL*TIMESTEP:.0f} ps equilibration
# Phase 2: {N_PROD*TIMESTEP:.0f} ps production (trajectory for NB11 MSD)
# Refs: NVT [6]; H diffusion [1,2]

units          metal
atom_style     atomic
newton         on
boundary       p p p

read_data      {BULK_H_FILE}

# ── Restart Settings ─────────────────────────────────────────
# Automatically save every 10,000 steps. 
# Using % ensures parallel I/O efficiency for GPU/MPI runs.
restart        10000 results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_{T}K.%.restart

pair_style     {PAIR_STYLE} {MACE_MODEL} {PAIR_SUFFIX}
pair_coeff     * * {ELEM_STR}

neighbor       2.0 bin
neigh_modify   every 1 delay 0 check yes

# ── Identify H atom for tracking ─────────────────────────────
group          H_atom   type 9
group          metal    subtract all H_atom

# ── Thermo ───────────────────────────────────────────────────
thermo         {THERMO_EVERY}
thermo_style   custom step temp pe ke etotal press vol

# ── Timestep ─────────────────────────────────────────────────
timestep       {TIMESTEP}

# ── Initialize velocities ────────────────────────────────────
velocity       all create {T}.0 42 mom yes rot yes dist gaussian

# ── Phase 1: Equilibration ───────────────────────────────────
fix            nvt_equil  all  nvt  temp  {T}.0  {T}.0  {TAU_T}
print "### Phase 1: Equilibration {N_EQUIL*TIMESTEP:.0f} ps at {T} K ###"
run            {N_EQUIL}
unfix          nvt_equil

# ── Phase 2: Production ──────────────────────────────────────
# Dump trajectory for NB11 MSD calculation
# Dump every {DUMP_EVERY} steps = {DUMP_EVERY*TIMESTEP*1000:.0f} fs
dump           prod_dump  all  custom  {DUMP_EVERY}  {traj_file} &
               id type x y z
dump_modify    prod_dump  sort id

fix            nvt_prod  all  nvt  temp  {T}.0  {T}.0  {TAU_T}

# Compute MSD for H atom
compute        msd_H  H_atom  msd
fix            msd_out  all  ave/time  1  1  {THERMO_EVERY} &
               c_msd_H[4]  file  results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_msd_H_{T}K.txt  mode scalar

print "### Phase 2: Production {N_PROD*TIMESTEP:.0f} ps at {T} K ###"
run            {N_PROD}

# ── Save final state ─────────────────────────────────────────
variable  pe_final  equal  pe
variable  temp_fin  equal  temp
variable  press_fin equal  press

print "EQUIL_RESULTS_START"
print "  T_K          : {T}"
print "  pe_final_eV  : ${{pe_final}}"
print "  temp_final_K : ${{temp_fin}}"
print "  press_final  : ${{press_fin}}"
print "EQUIL_RESULTS_END"
# ── Final Saves ──────────────────────────────────────────────
# Save a single binary file at the absolute end of the job
write_restart  results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_{T}K_final.restart
write_data     {out_file}
"""
    path = f'lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_{T}K.lammps'
    with open(path, 'w') as f:
        f.write(script)
    return path, log_file, out_file, traj_file

job_list = []
sc  = SLURM_CONFIG
kk  = ' '.join(KOKKOS_FLAGS)
ld  = '\n'.join(f'export LD_LIBRARY_PATH={p}:$LD_LIBRARY_PATH'
                for p in sc['ld_paths'])

for T in TEMPERATURES:
    script_f, log_f, out_f, traj_f = write_bulk_nvt(T)

    slurm = f"""#!/bin/bash
#SBATCH --job-name=Hastelloy_N_42equilibration_{T}K
#SBATCH --ntasks={sc['ntasks']}
#SBATCH --cpus-per-task={sc['cpus_per_task']}
#SBATCH --gres=gpu:{sc['gpu']}
#SBATCH --partition={sc['partition']}
#SBATCH --time={sc['time']}
#SBATCH --output=results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_slurm_bulk_{T}K_%j.out

module load OpenMPI/{sc['openmpi_ver']}
module load cuda/{sc['cuda_version']}
source ~/miniforge3/etc/profile.d/conda.sh
conda activate {sc['conda_env']}
{ld}
cd {os.getcwd()}

echo "T={T}K Start: $(date)"
{LAMMPS_CMD} {kk} -in {script_f} -log {log_f}
echo "T={T}K End: $(date)"
"""
    sp = f'slurm_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_run_bulk_{T}K.sh'
    with open(sp, 'w') as f:
        f.write(slurm)
    os.chmod(sp, 0o755)
    job_list.append((T, script_f, log_f, out_f, traj_f, sp))

print(f'Written {len(job_list)} LAMMPS + SLURM scripts')
for T, sf, lf, of, tf, sp in job_list:
    print(f'  {T} K: {sf}')
    print(f'        traj → {tf}')

Written 4 LAMMPS + SLURM scripts
  300 K: lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_300K.lammps
        traj → results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_300K_prod.lammpstrj
  400 K: lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_400K.lammps
        traj → results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_400K_prod.lammpstrj
  600 K: lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_600K.lammps
        traj → results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_600K_prod.lammpstrj
  800 K: lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_800K.lammps
        traj → results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_800K_prod.lammpstrj


In [4]:
# ─────────────────────────────────────────────────────────────
# Submit all 4 temperature jobs simultaneously
# 4 jobs × 1 A100 each — well within cluster limits
# All jobs are independent — no inter-job communication needed
# ─────────────────────────────────────────────────────────────

job_ids = {}

print(f'Submitting {len(job_list)} NVT jobs...')
print(f'{"T (K)":>8s}  {"Job ID"}')
print('-' * 25)

for T, sf, lf, of, tf, sp in job_list:
    result = subprocess.run(['sbatch', sp], capture_output=True, text=True)
    if result.returncode == 0:
        jid = result.stdout.strip().split()[-1]
        job_ids[T] = jid
        print(f'  {T:>6d} K  → Job {jid}')
    else:
        print(f'  {T:>6d} K  FAILED: {result.stderr.strip()[:50]}')

print(f'\n{len(job_ids)}/4 jobs submitted.')
print('Monitor: squeue -u $USER')
print(f'Each job: {(N_EQUIL+N_PROD)*TIMESTEP:.0f} ps total ({(N_EQUIL+N_PROD)*TIMESTEP/1000:.3f} ns)')
print('After all jobs finish, run cell 3.5.')

Submitting 4 NVT jobs...
   T (K)  Job ID
-------------------------
     300 K  → Job 6349714
     400 K  → Job 6349715
     600 K  → Job 6349716
     800 K  → Job 6349717

4/4 jobs submitted.
Monitor: squeue -u $USER
Each job: 1000 ps total (1.000 ns)
After all jobs finish, run cell 3.5.


In [5]:
# ─────────────────────────────────────────────────────────────
# Parse equilibration logs
# Verify: T reached, PE stable, no crashes
# ─────────────────────────────────────────────────────────────

def parse_equil_log(logfile):
    """Extract final T, PE, pressure from EQUIL_RESULTS block."""
    results = {}
    if not os.path.exists(logfile):
        return None
    with open(logfile) as f:
        for line in f:
            line = line.strip()
            for key in ['T_K', 'pe_final_eV', 'temp_final_K', 'press_final']:
                if key in line and ':' in line:
                    try:
                        results[key] = float(line.split(':')[1].strip())
                    except:
                        pass
    return results if results else None

def parse_thermo_series(logfile):
    """Read LAMMPS thermo output — return step, temp, pe arrays."""
    steps, temps, pes = [], [], []
    in_thermo = False
    if not os.path.exists(logfile):
        return None
    with open(logfile) as f:
        for line in f:
            line = line.strip()
            #if line.startswith('Step') and 'Temp' in line and 'PotEng' in line:
            if line.startswith('Step') and 'Temp' in line:
                in_thermo = True
                continue
            if in_thermo:
                if line.startswith('Loop') or line.startswith('WARNING'):
                    in_thermo = False
                    continue
                parts = line.split()
                if len(parts) >= 3:
                    try:
                        steps.append(float(parts[0]))
                        temps.append(float(parts[1]))
                        pes.append(float(parts[2]))
                    except:
                        pass
    if steps:
        return np.array(steps), np.array(temps), np.array(pes)
    return None

# Rebuild job_list if kernel restarted
job_list = []
for T in TEMPERATURES:
    job_list.append((
        T,
        f'lammps_scripts/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_nvt_{T}K.lammps',
        f'results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_equil_{T}K.log',
        f'structures/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_{T}K.lammps',
        f'results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_H_{T}K_prod.lammpstrj',
        f'slurm_scripts/notebook10-bulk-equilibration/N_42/run_Hastelloy_N_42_{T}K.sh'
    ))

print('Bulk Equilibration Results')
print('=' * 70)
print(f'{"T (K)":>6s} {"T_final (K)":>12s} {"PE (eV)":>14s} '
      f'{"Traj exists":>12s} {"Status"}')
print('-' * 70)

summary = {}
missing = []
all_ok  = True

for T, sf, lf, of, tf, sp in job_list:
    res = parse_equil_log(lf)
    traj_ok = os.path.exists(tf) and os.path.getsize(tf) > 0
    struct_ok = os.path.exists(of)

    if res:
        T_fin = res.get('temp_final_K', res.get('T_K', 0))
        pe    = res.get('pe_final_eV', 0)
        T_ok  = abs(T_fin - T) < 50  # within 50 K
        ok    = T_ok and traj_ok and struct_ok
        summary[T] = {'T_final': T_fin, 'pe': pe, 'ok': ok}
        status = '✓' if ok else 'WARN'
        print(f'  {T:>6d} {T_fin:>12.1f} {pe:>14.4f} '
              f'{"yes" if traj_ok else "NO":>12s} {status}')
        if not ok:
            all_ok = False
    else:
        missing.append(T)
        print(f'  {T:>6d} {"NOT FOUND":>12s}')
        all_ok = False

print('=' * 70)
if missing:
    print(f'⚠ {len(missing)} jobs not done: {missing}')
elif all_ok:
    print('All 4 temperatures equilibrated successfully.')

# Save summary
if summary:
    with open('results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_equil_summary.txt', 'w') as f:
        f.write('Hastelloy N Bulk Equilibration — Notebook 10\n')
        f.write('=' * 50 + '\n')
        f.write(f'System   : 500 metal + 1 H (octahedral site)\n')
        f.write(f'Protocol : {N_EQUIL*TIMESTEP:.0f} ps equil + {N_PROD*TIMESTEP:.0f} ps production\n\n')
        f.write(f'{"T_set":>8s} {"T_final":>10s} {"PE (eV)":>14s} {"OK"}\n')
        f.write('-' * 40 + '\n')
        for T, d in sorted(summary.items()):
            f.write(f'{T:>8d} {d["T_final"]:>10.1f} {d["pe"]:>14.4f} '
                    f'{"yes" if d["ok"] else "no"}\n')
    print('Saved: results/notebook10-bulk-equilibration/N_42/Hastelloy_N_42_equil_summary.txt')

Bulk Equilibration Results
 T (K)  T_final (K)        PE (eV)  Traj exists Status
----------------------------------------------------------------------
     300    NOT FOUND
     400    NOT FOUND
     600    NOT FOUND
     800    NOT FOUND
⚠ 4 jobs not done: [300, 400, 600, 800]
